# 智能入侵检测 —— 实时运动检测演示（PL 加速版）

**这是本工程优化后的正式演示。** 视频流经 FPGA 的 4 个 HLS IP 级联处理：

```
摄像头 -> PS -> AXI DMA -> rgb2gray -> frame_diff -> threshold -> morphology
                  <- AXI DMA <- 二值掩码
PS: 形态学二次处理 / 轮廓 / 合框 / 警戒区判断 / 显示 / 蜂鸣器
```

---

### 运行方法

1. 点上面菜单栏 **Cell → Run All**，或者
2. 选中下面的代码单元格，按 **Shift + Enter**

### 停止方法

点工具栏的 **■**（Interrupt Kernel 中断内核）。程序会自动关掉蜂鸣器、释放摄像头和 DMA 缓冲。

### 画面说明

| 元素 | 正常 | 报警 |
|---|---|---|
| 中间警戒区方框 | **蓝色** | **红色** |
| 左上角状态文字 | 绿色 `Normal` | 红色 `ALARM` |
| 运动目标框 | 绿色 | 绿色 |

左上角显示 `Targets: N`（当前目标数）和 `FPS`（帧率）。

In [ ]:
# 确保能 import 到 motion_common 等本地模块
import sys
sys.path.insert(0, "/home/xilinx/intrusion_demo")

# 运行实时演示（无限循环，用工具栏的 ■ 停止）
%run /home/xilinx/intrusion_demo/pl_motion_detection_optimized.py

Loading overlay once...
Overlay loaded


[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (616) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created


Image(value=b'', format='jpeg', height='480', width='640')

---

## 现场验收要点

1. **首帧不误报** —— 第一帧只用于初始化背景
2. 手在镜头前移动 → 出现**绿色目标框**，`Targets` 计数增加
3. 手伸进中间方框 → 方框**由蓝变红**、文字变 `ALARM`、**蜂鸣器间歇响**
4. 手停下不动 → 报警自动消失（帧差法的正常特性）
5. 停止程序后**蜂鸣器必须保持关闭**
6. 建议连续运行 3 分钟以上观察稳定性

## 出问题怎么办

| 现象 | 处理 |
|---|---|
| 红色 GStreamer warning 但画面正常 | 可忽略（交付文档已说明） |
| `DmaTransferError` / DMA 状态错误 | **立即中断**，保留报错原文 |
| 蜂鸣器一直响停不下来 | 立即中断；若无效直接拔板卡电源 |
| `RuntimeError: No Devices Found` | 说明 `XILINX_XRT` 环境缺失，见复现指南第 6.3 节 |
| 摄像头打不开 | 确认 USB 摄像头插在板卡 USB-A 口，Jupyter 以 root 运行 |

> 详细记录见 `E:\fpga\project\复现指南.md` 第 6 节。